[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/bending/bending.ipynb)

# Bending and Twisting of Thin-Walled Beams

This example computes how a beam bends and twists under a load, in Euler–Bernoulli beam theory and Saint-Venant's theory of torsion. As the beam bends, each of its cross-sections stays flat and turns a little about a line in its own plane, and the material on one side of that line is stretched while the material on the other side is squeezed. As it twists, each cross-section turns about a line along the beam. The beams here are thin-walled, like the steel sections of buildings and bridges, whose walls are thin compared with the section: that makes them stiff against bending and weak against twisting.

The computation is written in the projective geometric algebra of three-dimensional space, where a load is a line of force wherever it acts, a forque, and a strain, the motion from one cross-section to the next, is a twist, a line of motion: the same lines with the opposite orientation, an antibivector and a bivector. A fibre is the line along the beam through a point of the section, `beam_direction & point`, and a strain stretches it by `strain & fibre`. The section's stiffness is a sum of dyads of lines over its fibres and walls, a map from strains to loads, `Stiffness = Forque <- Twist`, which holds the area and the first and second moments of area with no centroid found first and no lever arms. One solve, `whole.solve(loads)`, takes any load line to the strain it causes, bending, twisting and sliding together. The shear centre is nowhere written into the stiffness: a twist settles on the line through it because there it costs no slide.

In the matrix notation of beam theory a cross-section's stiffness reads as a six-by-six matrix from the axial strain $\varepsilon$, the shear strains $\gamma_x, \gamma_y$, the curvatures $\kappa_x, \kappa_y$ and the rate of twist $\theta'$ to the stress resultants $N, V_x, V_y, M_x, M_y, T$, with entries such as $EA$, $EI_{xx}$, $EI_{xy}$, $GA_s$ and $GJ$, and couplings between them that depend on the reference point chosen.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, concatenate
from numga.algebras import PGA3D
from examples.util.animation import save_figure, show_animation
from examples.mechanics.bending import render
from examples.mechanics.bending.core import paths, point
from examples.mechanics.bending.mesh import NAMES, section_walls
from examples.mechanics.simplex import barycentric_samples, simplex_inertia_weights

In [ ]:
ga = PGA3D
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Plane = ga.gatype.vector()
Point = ga.gatype.antivector()
Twist = ga.gatype.bivector()
# Lines of force: loads, and the resultant of everything a section exerts.
Forque = ga.gatype.antibivector()
Motor = ga.gatype.rotor()
# Lines in the section's plane.
Strain = Twist.derive.containing("z")
# Lines along the beam, and couples across it.
Resultant = Forque.derive.within("x y w")
# The fibres' pulls weighted by a warping, a swept area signed by the sense of the sweep: resultants of the
# other orientation, and their couples across the beam.
Warped = Resultant.derive.flipped_orientation
Couples = Warped.derive.containing("w")
# The shifts of a pole across the beam: points at infinity in the section's plane.
Shift = ga.gatype.antivector().derive.containing("w z")
# A bending strain to the resultant of the fibres' pulls.
Bending = ga.gatype((Resultant, Strain))                   # Resultant <- Strain
# Any strain to the forque of everything the section exerts.
Stiffness = ga.gatype((Forque, Twist))                      # Forque <- Twist

# The cross-section at the clamp, the point at infinity along the beam, and the point at infinity up the
# section.
section_plane = mv.z                                       # [] Plane
beam_direction = section_plane.dual()                      # [] Point
up_direction = mv.y.dual()                                 # [] Point
# The line where the section's plane meets the plane at infinity: a turn about it shifts a section along the
# beam, the motion from one cross-section to the next of the unbent beam, per unit length.
shift_along_beam = section_plane ^ mv.w                    # [] Twist
# The same line as a load, a couple about the beam: the join of the directions across and up.
couple_about_beam = mv.x.dual() & mv.y.dual()              # [] Forque

In [ ]:
# The example cross-sections: the nodes of each one's midline, its walls as pairs of nodes, and the walls'
# thickness.
sections = section_walls()

## 1. The stiffness of a cross-section

The beam is modelled as a bundle of fibres: a fibre is a point of the cross-section extruded along the length of the beam, and each fibre stretches or shortens on its own. When the beam bends, each cross-section stays flat and turns a little, relative to its neighbour, about a line in its own plane, the neutral axis; the fibres on one side of that line are stretched and those on the other side squeezed, in proportion to their distance from it.

In projective geometric algebra a turn about a line is described by the line itself, with the line's size the rate of turning, and a line at infinity describes a shift instead of a turn. How one cross-section moves relative to the next, per unit length of beam, is therefore a single line in the section's plane, `Strain`, sized by how fast the sections turn about it: the neutral axis. A shift along the beam, which stretches every fibre alike, moves that line within the plane, and a pure shift puts it at infinity. This line is the strain of the section.

In the algebra a fibre is the line along the beam through its point, `beam_direction & point`, and the regressive product `strain & fibre` is its stretch: zero for a fibre that meets the neutral axis, and growing with its distance from it. A stretched fibre pulls along itself in proportion to its stretch, and over `Strain`, the space of all strains, that pull is a map, `modulus * fibre * (Strain & fibre)`; summed over the section it is the section's stiffness in bending, from a strain to the resultant of all the fibres' pulls, their combined pull as a single line of force.

The sections here are thin-walled: each is a set of walls, straight pieces of a midline with a thickness, and the sum over the fibres is an integral along the walls, each a strip whose area is its length times its thickness. It is computed by an exact quadrature for second moments over a simplex, the same in any number of dimensions: a wall is a simplex of one dimension, and it has the same second moment as two equal point masses on it, as the [simplex example](../simplex.py) computes and checks against sampling.

In [ ]:
# The material's modulus.
modulus = 1.0

In [ ]:
def lengths(walls: Point) -> Scalar:
    """Each wall's length, the size of the line through its two ends."""
    return (walls[..., 0] & walls[..., 1]).norm()


def samples(walls: Point, thickness: np.ndarray) -> Point:
    """Two points along each wall, each with the square root of its share of the wall's area."""
    weights = simplex_inertia_weights(2)
    return barycentric_samples(weights, walls) * (lengths(walls) * thickness).square_root()[..., None]


def bending(walls: Point, thickness: np.ndarray, modulus: float) -> Bending:
    """Sum the fibres' pulls under a bending strain, integrating each wall exactly."""
    fibres = beam_direction & samples(walls, thickness)                     # [walls, 2] Resultant
    stretches = Strain & fibres                                             # [walls, 2] Scalar <- Strain
    # Each point carries the square root of its share of the area and appears twice, so its term counts
    # that share once.
    tensions = modulus * fibres * stretches                                 # [walls, 2] Resultant <- Strain
    return tensions.sum(axis=(-2, -1))                                      # [] Resultant <- Strain


nodes, ends, thickness = sections[NAMES.index("Z-section")]                 # [nodes] Point, [walls, 2], [walls]
bending(nodes[ends], thickness, modulus)

The Z-section's bending stiffness holds its area and its first and second moments of area, all from the one sum, with no centroid found first and no lever arms. Its input, `Strain`, is the lines in the section's plane; its output, `Resultant`, is the lines along the beam, the pulls of the fibres, and the couples across it, pure turning effects, which a pair of opposite pulls on parallel fibres makes.

## 2. Twisting and the shear centre

A beam twists when a load turns it about its own length, and a thin-walled beam with an open section, one whose walls do not close around a hollow, twists easily. In Saint-Venant's theory of torsion, the resistance of open walls to a rate of twist is the material's shear modulus times a third of each wall's length times the cube of its thickness, summed over the walls: for thin walls a small number.

Each cross-section twists about a line along the beam, and the line is decided by how the walls warp. Under a twist the walls do not stay in the section's plane: each point of a wall moves along the beam in proportion to its warping, the area that the line from the twisting axis sweeps as it follows the walls from a fixed starting point to that point. The twisting axis passes through the shear centre, the point whose warping no bending can see: the fibres, stretched in proportion to the warping, pull with no resultant couple and no resultant pull across the section. So twisting about the shear centre does not bend the beam, and a load through the shear centre bends it without twisting it.

In the algebra, the area swept from a pole along a wall from one end to the other is the join of the point at infinity along the beam with the pole and the wall's two ends, `beam_direction & pole & start & end`, a signed area that needs no coordinates, signed by the sense of the sweep. Added up along the walls from node to node, it is the warping at each node, and along each wall the warping runs linearly between its ends. Each fibre weighted by its warping and summed, as for the bending stiffness, gives a line, the warping's resultant; signed by the sweep, it has the other orientation from the fibres' own pulls. The warping is linear in the pole: moving the pole from the centroid by a shift adds the warping seen from the shift itself, a map from shifts to resultants. The shear centre is the pole whose warping pulls with no couple across the beam, and one solve of that map against the couples seen from the centroid finds the step to it.

In the notation of thin-walled beam theory the warping is the sectorial coordinate $\omega$, and the shear centre is the pole about which the sectorial products of area $\int \omega\, x\, \mathrm{d}A$ and $\int \omega\, y\, \mathrm{d}A$ vanish.

In [ ]:
# The material's Poisson's ratio, and its shear modulus.
poisson = 0.3
shear_modulus = modulus / (2 * (1 + poisson))

In [ ]:
def centroid(walls: Point, thickness: np.ndarray) -> Point:
    """The centroid of the whole section: the middle of each wall, weighted by the wall's area."""
    return ((walls[..., 0] + walls[..., 1]) * (lengths(walls) * thickness / 2)).sum(axis=-1).normalized()


def warping(nodes: Point, ends: np.ndarray, thickness: np.ndarray, pole: Point) -> Warped:
    """The fibres weighted by their warping seen from the pole, summed over the walls."""
    walls = nodes[ends]                                                     # [walls, 2] Point
    swept = beam_direction & pole & walls[..., 0] & walls[..., 1]           # [walls] Scalar
    # The swept areas added up along the walls to each node, up to one constant for all nodes.
    warped = (swept * paths(ends, nodes.shape[0])).sum(axis=-1)            # [nodes] Scalar
    shares = (lengths(walls) * thickness).square_root()[..., None]          # [walls, 1] Scalar
    along = barycentric_samples(simplex_inertia_weights(2), warped[ends]) * shares   # [walls, 2] Scalar
    return (along * (beam_direction & samples(walls, thickness))).sum(axis=(-2, -1))  # [] Warped


def shear_centre(nodes: Point, ends: np.ndarray, thickness: np.ndarray) -> Point:
    """The pole whose warped fibres pull with no couple across the beam: the centroid, moved by the shift
    whose own warping cancels the centroid's couples."""
    walls = nodes[ends]                                                     # [walls, 2] Point
    centre = centroid(walls, thickness)                                     # [] Point
    couples_per_shift = warping(nodes, ends, thickness, Shift).cast(Couples)   # [] Couples <- Shift
    centroid_couples = warping(nodes, ends, thickness, centre).cast(Couples)   # [] Couples
    return centre - couples_per_shift.solve(centroid_couples)                  # [] Point


def twisting(walls: Point, thickness: np.ndarray, shear_modulus: float) -> Scalar:
    """The torque per rate of twist of the open walls: a third of each wall's length times the cube of
    its thickness."""
    return shear_modulus * (lengths(walls) * thickness**3 / 3).sum(axis=-1)    # [] Scalar


centres = [centroid(nodes[ends], thickness) for nodes, ends, thickness in sections]   # [] Point each
shear_centres = [shear_centre(*section) for section in sections]            # [] Point each
save_figure(render.draw_walls(sections, centres, shear_centres), "bending_walls");

The walls are drawn along their midlines, with the centroid as a dot and the shear centre as a ring. The I-sections and the Z-section are each unchanged by a half turn about their centroid, so their shear centres lie on their centroids. The slotted tube's shear centre lies outside the tube, on the side opposite its slot: a load through the tube's centroid passes beside it, and twists the tube.

## 3. The whole stiffness

A cross-section can move relative to the next in six ways, one for each component of a line: it can shift along the beam, turn about two lines in its own plane, twist about a line along the beam, and slide across the beam in two directions. The fibres resist the shift and the turns, and the walls resist the twist, as above. The walls also resist the slides, which Timoshenko's theory of beams adds to Euler–Bernoulli's: each wall resists the section sliding along its own direction, by the shear modulus times the wall's area. All three together make the whole stiffness of the section, a map from any strain line to the forque of everything the section exerts, `Stiffness = Forque <- Twist`, and each of its parts is a sum of dyads of lines.

In the algebra, the twist's part is a dyad of the line at infinity of the section's plane. As a strain that line, `shift_along_beam`, shifts the section along the beam; as a load, `couple_about_beam`, the join of the directions across and up, it is a couple about the beam, a torque; and paired with any strain the couple gives the strain's rate of twist, whatever line the twist is about. Each slide's part is a dyad of a force along the wall's direction through the shear centre, `shear_centre & direction`, which pairs with a slide in that direction. A twist about the line along the beam through the shear centre meets every one of these forces, and pairs with none of them: twisting about the shear centre costs only the twist, while a twist about any other line costs a slide as well. The twisting axis is nowhere written into the stiffness; it comes out of the solve.

Euler–Bernoulli's theory takes the slides to be impossible, a shear stiffness without bound, which no finite map can hold. With the walls' shear stiffness the map is finite, a load across the beam slides each section a little as well as bending it, and Euler–Bernoulli bending is the limit as the shear stiffness grows.


In [ ]:
def stiffness(nodes: Point, ends: np.ndarray, thickness: np.ndarray, modulus: float, shear_modulus: float) -> Stiffness:
    """The whole stiffness of the section, a sum of dyads of lines."""
    walls = nodes[ends]                                                     # [walls, 2] Point
    # The fibres' pulls, for the shift along the beam and the turns.
    fibres = beam_direction & samples(walls, thickness)                     # [walls, 2] Resultant
    pulling = (modulus * fibres * (Twist & fibres)).sum(axis=(-2, -1))       # [] Resultant <- Twist
    # The torque about the beam, for the twist.
    twisted = twisting(walls, thickness, shear_modulus) * couple_about_beam * (couple_about_beam & Twist)   # [] Forque <- Twist
    # A force along each wall through the shear centre, for a slide along the wall.
    along = (walls[..., 1] - walls[..., 0]) / lengths(walls)                # [walls] Point: each wall's direction
    slides = shear_centre(nodes, ends, thickness) & along                   # [walls] Forque
    sliding = (shear_modulus * lengths(walls) * thickness * slides * (slides & Twist)).sum(axis=-1)   # [] Forque <- Twist
    return pulling + twisted + sliding                                      # [] Forque <- Twist


stiffness(*sections[NAMES.index("slotted tube")], modulus, shear_modulus)

## 4. Cantilevers under a load at the tip

Each beam is a cantilever: clamped at one end, `z = 0`, and loaded straight down at the other end, its tip, through the centroid of the section there. The beam is cut into pieces along its length, and the strain is computed at the middle of each piece.

A cross-section holds up everything between it and the tip, so it must carry the load as seen from that section. In the algebra the load is a line, through the tip along the load's direction, and seeing it from a section means shifting the line back along the beam by the section's distance from the clamp, so that the section lies in the plane of the clamp. The shifted line holds everything the section must carry: for a load across the beam, a couple, the load times its distance from the section, which the fibres carry; the force across the beam, which the slides carry; and, where the line misses the shear centre, a moment about the beam, which the twist carries. The whole stiffness solves for the strain that carries all of it, in one solve for every station.

The strains then bend and twist the beam. An unbent beam carries each cross-section to the next by a shift along the beam, and a shift is a turn about a line at infinity: here the line where the section's plane meets the plane at infinity, `shift_along_beam = section_plane ^ w`. The strain adds to it. Within a piece, each cross-section moves relative to the next, per unit length, by `shift_along_beam + strain`: shifted along the beam, turned about the neutral axis, twisted about the line through the shear centre, and slid across. Over the length of the piece this motion adds up to a rigid motion, its exponential, a motor; the motors of the pieces, multiplied in turn from the clamp outwards, place every cross-section of the bent beam.

The couples and moments are measured on the straight beam, which holds for small deflections and twists, and the clamp lets the walls warp freely.

In [ ]:
# The beam's length, the pieces it is cut into, and the load at its tip.
length = 10.0
n_pieces = 40
load = 1e-5

In [ ]:
def loads_at_stations(loads: Forque, stations: np.ndarray) -> Forque:
    """The loads seen from each station along the beam, as if the station were the clamp."""
    shifts = (shift_along_beam * (stations / 2)).exp()                       # [stations] Motor
    return shifts << loads[..., None]                                        # [..., stations] Forque


def bend(strains: Twist, step: float) -> Motor:
    """The frame of each cross-section from the clamp to the tip `[..., pieces + 1]`."""
    # Integrate each constant strain exactly; this does not update the load on the bent geometry.
    piece_motors = ((shift_along_beam + strains) * (step / 2)).exp()               # [..., pieces] Motor
    clamp = mv.rotor().broadcast_to(piece_motors.shape[:-1] + (1,))                # [..., 1] Motor
    # Each later motor multiplies from the right: the reverse of the running product of the reverses.
    return concatenate([clamp, piece_motors.reverse().cumprod(axis=-1).reverse()], axis=-1)   # [..., pieces + 1] Motor


def stress(strains: Twist, modulus: float, points: Point) -> Scalar:
    """The stress along the beam at points of the section: the modulus times each fibre's stretch."""
    return modulus * (strains & (beam_direction & points))                   # [...] Scalar


def tip(points: Point, length: float) -> Point:
    """Points of the section carried to the tip of the unbent beam."""
    return (shift_along_beam * (length / 2)).exp() >> points


def cantilever(nodes: Point, ends: np.ndarray, thickness: np.ndarray, loads: Forque, length: float,
               shear_modulus: float) -> tuple[Twist, Motor, Strain, Scalar]:
    """A cantilever of a length under load lines at its tip, batched over loads: the strain of each piece, the
    frame of each cross-section, the principal bending axes, and the stress at the clamp at the walls' ends."""
    step = length / n_pieces
    # The middle of each piece.
    stations = (np.arange(n_pieces) + 0.5) * step                             # [pieces]
    walls = nodes[ends]                                                     # [walls, 2] Point
    rigidity = bending(walls, thickness, modulus)                            # [] Resultant <- Strain
    whole = stiffness(nodes, ends, thickness, modulus, shear_modulus)        # [] Forque <- Twist
    # Lever arms are measured on the straight reference beam. A strain moves the frames by
    # `exp(strain * step / 2)`, against its own sense, so it is solved against the load turned in sign.
    strains = whole.solve(-loads_at_stations(loads, stations))              # [..., pieces] Twist
    frames = bend(strains, step)                                             # [..., pieces + 1] Motor
    # Principal bending axes pass through the centroid; the metric does not see the shift along the beam.
    _, axes = (Strain | Strain).eigh(Strain & rigidity)                      # [3] Strain
    stresses = stress(strains[..., 0, None, None], modulus, walls)           # [..., walls, 2] Scalar
    return strains, frames, axes[:2], stresses


down = load * -up_direction                                                   # [] Point
beams = [cantilever(*section, tip(centre, length) & down, length, shear_modulus) for section, centre in zip(sections, centres)]
save_figure(render.draw_beams(sections, centres, shear_centres, down, beams), "bending_sections");

The top row shows each section at the clamp, coloured by the stress along the beam, with the tip's cross-section in blue where the load has moved it: the load in orange, the neutral axis in green, and, dashed through the centroid, the principal bending axes, the two directions in which a load bends the beam straight along itself. The bottom row shows the bent beams beside the straight ones, with the line of shear centres dashed. The I-section is mirror-symmetric about the vertical, so its neutral axis lies square to the load and its tip moves straight down. The turned I-section and the Z-section have no such symmetry: their neutral axes tilt, and their tips move down and sideways, towards the direction in which the section bends most easily. The slotted tube bends straight down, but its load passes beside its shear centre, and the tube twists as well.

In the matrix notation of beam theory the curvatures read as $\begin{pmatrix}\kappa_x\\ \kappa_y\end{pmatrix} = \frac{1}{E}\begin{pmatrix}I_{yy} & -I_{xy}\\ -I_{xy} & I_{xx}\end{pmatrix}^{-1}\begin{pmatrix}M_x\\ M_y\end{pmatrix}$, with $I_{xy}$ the product moment of area, which a mirror symmetry makes zero, the rate of twist as $\theta' = T / (G J)$, with $J = \sum b\, t^3 / 3$ over the walls and $T$ the torque about the shear centre, and the slide as $\gamma = V / (G A_s)$, with $A_s$ the shear area and $V$ the force across the beam.

## 5. A stubby beam

On the long beams above, the slides add only a few percent to the deflection. On a stubby beam, about twice as long as it is deep, they add nearly as much as the bending does. The I-beam below is that short, drawn twice: solid blue with the walls' shear stiffness, and translucent grey with a shear modulus a million times larger, the slides held practically rigid, as Euler–Bernoulli's theory takes them.

Without slides, each cross-section stays square to the bent beam, and the flanges leave the clamp level. With them, each cross-section also slides down relative to its neighbour, without turning: the flanges leave the clamp already sloping down while the cross-sections there stand upright, and the tip hangs lower by the slide, the force across the beam over the shear stiffness of the web.


In [ ]:
# The stubby beam's length and load, and a shear modulus a million times the material's.
stubby_length = 2.0
stubby_load = 1.5e-3 * -up_direction                                          # [] Point
rigid_shear = 1e6 * shear_modulus

In [ ]:
i_section = NAMES.index("I-section")
stubs = [cantilever(*sections[i_section], tip(centres[i_section], stubby_length) & stubby_load, stubby_length, shear)[1]
         for shear in (shear_modulus, rigid_shear)]                           # [pieces + 1] Motor each
save_figure(render.draw_stubby(sections[i_section], stubs, centres[i_section], stubby_load), "bending_stubby");

## 6. The Z-section with its load turning around

The load on the Z-section's tip turns once around the beam, in the plane of the section. Along either principal bending axis the tip moves along the load; between them it moves towards the axis about which the section bends most easily. Over the whole turn the tip's cross-section goes around an ellipse whose axes are the principal axes. The load always passes through the shear centre, which for the Z-section is the centroid, and the beam does not twist.

In [ ]:
# The frames of the load's turn, and the duration of each.
n_frames = 48
duration_ms = 60

In [ ]:
angles = np.linspace(0, 2 * np.pi, n_frames, endpoint=False)             # [frames]
loads = (mv.xy * (-angles / 2)).exp() >> down                               # [frames] Point
z_section = NAMES.index("Z-section")
turning = cantilever(*sections[z_section], tip(centres[z_section], length) & loads, length, shear_modulus)
show_animation(render.animate_loads(sections[z_section], centres[z_section], shear_centres[z_section], centres[z_section], loads, turning),
               "bending_turning", duration_ms)

## 7. The slotted tube with its load moved across

The load on the slotted tube's tip moves across the section along its line of symmetry, from the side of the slot to beyond the shear centre, always pointing down. Its couple about every section stays the same, and so does the bending; its moment about the twisting axis shrinks as the load nears the shear centre and changes sign as it passes. The tube twists one way while the load is on the slot's side of the shear centre, not at all as the load passes through it, and the other way beyond, as the tip's cross-section, in blue, turns about the ring.

In [ ]:
# The ends of the load's move across, across and up.
start, end = point([[0.4, 0.0], [-1.15, 0.0]])                              # [] Point each

In [ ]:
fractions = np.linspace(0, 1, n_frames)                                  # [frames]
across = start * (1 - fractions) + end * fractions                           # [frames] Point
tube = NAMES.index("slotted tube")
moved = cantilever(*sections[tube], tip(across, length) & down, length, shear_modulus)
show_animation(render.animate_loads(sections[tube], centres[tube], shear_centres[tube], across, down, moved),
               "bending_across", duration_ms)

## 8. Pushes along the beam, and the kern

A push along the beam through a point of the section is the line `beam_direction & point`, which is already a resultant, so the bending stiffness solves for its strain directly. A push through the centroid squeezes every fibre alike, and its neutral axis is the line at infinity. A push off the centroid also bends the section, and its neutral axis comes in from infinity on the far side of the centroid; where that line crosses the section, the fibres beyond it are stretched. The points through which a push leaves every fibre squeezed form the kern of the section. Solving the bending stiffness takes each push point to its neutral axis, a pairing of the points and lines of the section's plane called a polarity, and the pairing is symmetric: a pull through one point stretches the fibre through a second as much as a pull through the second stretches the fibre through the first.

That symmetry constructs the kern. The pushes that leave the fibre through one node unstretched are exactly the points of that node's polar, so the kern is the region on the centroid's side of the polar of every node. Only the corners of the section's convex hull matter: their polars are the kern's edges, and the poles of the hull's edges, the push points whose neutral axes run along them, are the kern's corners. The kern is the polar of the section's convex hull.

In the notation of classical strength of materials the neutral axis of a push at $(e_x, e_y)$ in principal axes reads $1 + \frac{e_x x}{i_y^2} + \frac{e_y y}{i_x^2} = 0$, with $i_x, i_y$ the radii of gyration: the antipolar of the push point with respect to the central ellipse of inertia.

In [ ]:
# The size of a push along the beam, and the points it goes through, across and up: one inside every
# section's kern and one outside.
push = 1e-3
points = point([[0.03, 0.04], [0.1, 0.3]])                                   # [pushes] Point

In [ ]:
def pulls(bending: Bending, points: Point) -> Strain:
    """The strain of a unit pull along the beam through each point of the section: its neutral axis,
    the point's polar line."""
    return bending.solve(beam_direction & points)                            # [...] Strain


def push_section(nodes: Point, ends: np.ndarray, thickness: np.ndarray, points: Point) -> tuple[Strain, Scalar, Strain]:
    """Pushes along the beam through points of a section: their strains and the stresses at the ends of
    the walls; and the polar of every node, which bounds the kern."""
    walls = nodes[ends]                                                     # [walls, 2] Point
    rigidity = bending(walls, thickness, modulus)                            # [] Resultant <- Strain
    strains = -push * pulls(rigidity, points)                                # [pushes] Strain
    stresses = stress(strains[..., None, None], modulus, walls)              # [pushes, walls, 2] Scalar
    return strains, stresses, pulls(rigidity, nodes)


pushes = [push_section(*section, points) for section in sections]
save_figure(render.draw_pushes(sections, centres, points, pushes), "bending_kern");

The kern is drawn in purple, the push as a cross, and its neutral axis in green; the top row pushes through a point inside every kern, the bottom row through a point outside. Each kern is cut out of the section's plane by the polars of the nodes: a thin diamond for the I-sections, whose hulls are rectangles, a slanted quadrilateral for the Z-section, and for the slotted tube a polygon of many short edges, one for each corner of the arc, with a corner on the side away from the slot, the pole of the chord across it. The kern of the slotted tube lies in its hollow, where a push can act through a plate on the beam's end.